# 01 — NMF global das frases (modelo final)

Primeiro estágio do modelo final de sentenças: NMF (scikit-learn, TF-IDF) sobre a **frase
central** de todas as frases filtradas, com K fixo em `params.yaml::final_sentence_pipeline.global_k`.

- **Entrada:** `02-sentences/data/output/youtube_sent/youtube_sent_*/corpus_sentencas.csv` (o mais recente).
- **Saída:** `03-topic-modeling/data/output/youtube_sent/nmf_global/nmf_global_sentence_k{K}_{data}/`
  com `nmf_results.csv`, `topics.csv`, `topics.json`, `topic_weights.npz`, `model.pkl` e `manifest.json`.
- **Depois deste notebook:** ler os temas, escolher os temas de aspecto e preencher
  `global_run`, `selected_topics` e `expected_sentences` no `params.yaml` (última seção).
  Em seguida, rodar `02_nmf_topicos.ipynb`.

O código do modelo está em `03-topic-modeling/scripts/run_global_nmf_sentences.py`; este
notebook só o chama e mostra os resultados. Os pesos são ativações NMF normalizadas por frase,
não probabilidades calibradas.

In [1]:
import json, sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "03-topic-modeling").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "03-topic-modeling" / "scripts"))
pd.set_option("display.max_colwidth", 200)

PARAMS_PATH = ROOT / "03-topic-modeling" / "configs" / "params.yaml"
params = yaml.safe_load(PARAMS_PATH.read_text(encoding="utf-8"))
FINAL = params["final_sentence_pipeline"]
SEED = params["seed"]

import run_global_nmf_sentences as nmf_global

K = FINAL["global_k"]
print("K global:", K, "| seed:", SEED)

K global: 20 | seed: 42


## 1. Treinar

`RUN_EXISTENTE = None` treina um run novo. Para só reabrir um run já treinado, coloque o nome
da pasta (ex.: `"nmf_global_sentence_k20_20261006_101500"`).

In [2]:
RUN_EXISTENTE = FINAL["global_run"]   # None = treinar um run novo

if RUN_EXISTENTE:
    RUN = nmf_global.OUTPUT_ROOT / RUN_EXISTENTE
else:
    fonte = nmf_global.source_file()
    print("Fonte:", fonte)
    RUN = nmf_global.run(fonte, [K], SEED, max_features=12000, text_column="sentence")[0]
print("Run:", RUN.name)

Run: nmf_global_sentence_k20_20261005_221423


In [3]:
manifest = json.loads((RUN / "manifest.json").read_text(encoding="utf-8"))
pd.Series(manifest).to_frame("valor")

,valor
source,D:\Documentos\youtube_metadata_collection\02-sentences\data\output\youtube_sent\youtube_sent_20261005_214426\corpus_sentencas.csv
source_size,3262848279
source_modified,1791249250.00511
seed,42
k,20
text_column,sentence
target_column,sentence
min_df,20
max_df,0.5
max_features,12000


## 2. Temas: termos e tamanho

In [4]:
topics = pd.read_csv(RUN / "topics.csv")
res = pd.read_csv(RUN / "nmf_results.csv", dtype={"sent_id": str, "post_id": str})
videos = res.loc[res.topic_id >= 0].groupby("topic_id").post_id.nunique().rename("n_videos")
topics = topics.join(videos, on="topic_id")
topics["pct_frases"] = (100 * topics.n_sentences / len(res)).round(1)
print(f"{len(res)} frases; {int((res.topic_id == -1).sum())} sem termos no vocabulário (tema -1)")
topics[["topic_id", "n_sentences", "pct_frases", "n_videos", "top_terms"]]

116111 frases; 805 sem termos no vocabulário (tema -1)


,topic_id,n_sentences,pct_frases,n_videos,top_terms
0,0,3531,3.0,918,"just, doesn, feels, feel, device, way, make, work, time, looks, product, regular, people, isn, mean, doing, point, using, apple, things"
1,1,6104,5.3,1020,"good, pretty, quality, speakers, sound, build, looks, overall, trackpad, job, quite, price, actually, device, news, camera, color, selection, surprisingly, deal"
2,2,5768,5.0,812,"laptop, gaming, best, price, buy, looking, surface, light, premium, portable, powerful, inch, use, buying, speakers, processor, ports, display, windows, warm"
3,3,4064,3.5,959,"battery, life, hours, hour, watt, long, getting, great, day, milliamp, big, longer, use, got, half, smaller, test, power, bigger, size"
4,4,3655,3.1,953,"don, know, need, love, things, people, feel, use, care, buy, mind, sound, personally, maybe, did, way, run, recommend, worry, loud"
5,5,3854,3.3,957,"really, nice, cool, actually, looks, device, built, does, big, fast, feel, didn, solid, mean, doesn, looking, feels, need, camera, works"
6,6,4808,4.1,757,"pro, macbook, air, inch, chip, max, apple, ipad, pros, core, mac, surface, base, yoga, does, iphone, model, display, buy, slim"
7,7,4746,4.1,985,"little, bit, different, actually, feel, does, things, feels, bigger, quite, maybe, weird, smaller, looks, thicker, fan, warm, noise, flex, faster"
8,8,5065,4.4,633,"phone, actually, camera, pretty, great, big, using, things, android, google, time, pixel, samsung, phones, oneplus, flagship, galaxy, best, display, basically"
9,9,4337,3.7,1006,"right, left, device, usb, ports, port, button, best, got, market, hand, power, jack, keys, tell, headphone, bat, let, buy, thunderbolt"


## 3. Temas por categoria de produto

Percentual das frases de cada tema por categoria. Compare com a linha **base** (todas as
frases): temas de aspecto compartilhado acompanham a base; temas de categoria se concentram.
Lembre que canal e categoria se confundem neste corpus (ver idealização §5.3C).

In [5]:
validas = res.loc[res.topic_id >= 0]
tab = pd.crosstab(validas.topic_id, validas.product_category, normalize="index").mul(100)
base = validas.product_category.value_counts(normalize=True).mul(100).rename("base")
cols = base.index[:6]
pd.concat([tab[cols], base[cols].to_frame().T]).round(0)

product_category,laptop,phone,other,tablet,headphone,watch
0,49.0,26.0,8.0,3.0,4.0,4.0
1,55.0,28.0,4.0,3.0,4.0,2.0
2,85.0,8.0,2.0,1.0,0.0,1.0
3,57.0,28.0,3.0,3.0,3.0,3.0
4,49.0,29.0,6.0,3.0,4.0,3.0
5,48.0,32.0,5.0,4.0,4.0,2.0
6,71.0,12.0,2.0,7.0,2.0,0.0
7,50.0,31.0,5.0,3.0,4.0,2.0
8,8.0,78.0,5.0,2.0,3.0,3.0
9,57.0,25.0,6.0,3.0,3.0,1.0


## 4. Frases exemplares

Para cada tema: frases de maior peso (vídeos distintos) e frases sorteadas. Leia as duas
listas: o peso alto tende a favorecer frases curtas e descritivas.

In [6]:
exemplos = json.loads((RUN / "topics.json").read_text(encoding="utf-8"))
for t in exemplos:
    print(f"\n=== Tema {t['topic_id']} ({t['n_sentences']} frases) — {t['top_terms'][:110]}")
    print("  maior peso:")
    for e in t["strong_examples"][:4]:
        print(f"    [{e['weight']:.2f}|{e['category']}] {e['sentence'][:150]}")
    print("  sorteadas:")
    for e in t["random_examples"][:3]:
        print(f"    [{e['weight']:.2f}|{e['category']}] {e['sentence'][:150]}")


=== Tema 0 (3531 frases) — just, doesn, feels, feel, device, way, make, work, time, looks, product, regular, people, isn, mean, doing, po
  maior peso:
    [1.00|headphone] yeah I just totally picked them up
    [1.00|watch] But again, that's just promised.
    [1.00|phone] But yeah, it's just I can trust my eye about what I observe.
    [1.00|headphone] So that's not a criticism, that's just context.
  sorteadas:
    [0.28|watch] So basically, find my iPhone before on the watch was just pinging a sound to play from your iPhone.
    [0.52|other] So yeah, I'd recommend this spark to just about anyone who's getting into entry level XLR microphones.
    [0.68|laptop] the blade 14 and asus g14 just crushes it.

=== Tema 1 (6104 frases) — good, pretty, quality, speakers, sound, build, looks, overall, trackpad, job, quite, price, actually, device, 
  maior peso:
    [1.00|phone] What's so good about it?
    [1.00|headphone] These are still very good.
    [1.00|phone] well it's still not tha

## 5. Escolher os temas de aspecto e fixar no `params.yaml`

A escolha é qualitativa: temas que descrevem aspectos do produto (bateria, tela, teclado...),
não temas de fala genérica. Liste os IDs em `TEMAS` e copie o bloco impresso para
`params.yaml::final_sentence_pipeline`. Os IDs só valem para este run.

In [7]:
TEMAS = FINAL["selected_topics"]   # temas de aspecto escolhidos (ver params.yaml)

n_sel = int(res.topic_id.isin(TEMAS).sum())
print(f"{len(TEMAS)} temas, {n_sel} frases ({100 * n_sel / len(res):.1f}% das frases)\n")
print("Copie para params.yaml::final_sentence_pipeline:")
print(f"  global_run: {RUN.name}")
print(f"  selected_topics: {sorted(TEMAS)}")
print(f"  expected_sentences: {n_sel}")

10 temas, 57952 frases (49.9% das frases)

Copie para params.yaml::final_sentence_pipeline:
  global_run: nmf_global_sentence_k20_20261005_221423
  selected_topics: [1, 3, 5, 7, 9, 10, 11, 14, 15, 17]
  expected_sentences: 57952
